In [1]:
#https://drive.google.com/file/d/1CppeqGbiBzX61jx56gcXyGGDNEeLlJ-W/view?usp=share_link
!gdown 1CppeqGbiBzX61jx56gcXyGGDNEeLlJ-W

Downloading...
From: https://drive.google.com/uc?id=1CppeqGbiBzX61jx56gcXyGGDNEeLlJ-W
To: /content/train_loan_imbalanced.csv
100% 38.0k/38.0k [00:00<00:00, 59.6MB/s]


In [61]:
import pandas as pd
df = pd.read_csv('/content/train_loan_imbalanced.csv')
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [62]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import SMOTENC
from sklearn.model_selection import train_test_split

In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [51]:
df = pd.read_csv('/content/train_loan_imbalanced.csv')

df_clean = df.copy()

df_clean = df_clean.drop('Loan_ID', axis=1)
df_clean['Gender'] = np.where(df_clean['Gender'] == 'Male', 1, 0)
df_clean['Married'] = np.where(df_clean['Married'] == 'No', 1, 0)
df_clean['Education'] = np.where(df_clean['Education'] == 'Graduate', 1, 0)
df_clean['Self_Employed'] = np.where(df_clean['Self_Employed'] == 'Yes', 1, 0)
df_clean['Property_Area'] = np.where(df_clean['Property_Area'] == 'Urban', 1, 0)
df_clean['Loan_Status'] = np.where(df_clean['Loan_Status'] == 'Y', 1, 0)

df_clean['Loan_Amount_Term'] = df_clean['Loan_Amount_Term'].fillna(df_clean['Loan_Amount_Term'].mean())

df_clean['Dependents'] = pd.to_numeric(df_clean['Dependents'], errors='coerce')
df_clean['Dependents'] = df_clean['Dependents'].fillna(df_clean['Dependents'].mode())

df_clean['LoanAmount'] = df_clean['LoanAmount'].fillna(df_clean['LoanAmount'].mean())
df_clean['Credit_History'] = df_clean['Credit_History'].fillna(df_clean['Credit_History'].mode())
df_clean.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,1,1,0.0,1,0,5849,0.0,146.412162,360.0,1.0,1,1
1,1,0,1.0,1,0,4583,1508.0,128.000000,360.0,1.0,0,0
2,1,0,0.0,1,1,3000,0.0,66.000000,360.0,1.0,1,1
3,1,0,0.0,0,0,2583,2358.0,120.000000,360.0,1.0,1,1
4,1,1,0.0,1,0,6000,0.0,141.000000,360.0,1.0,1,1


In [52]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gender             614 non-null    int64  
 1   Married            614 non-null    int64  
 2   Dependents         548 non-null    float64
 3   Education          614 non-null    int64  
 4   Self_Employed      614 non-null    int64  
 5   ApplicantIncome    614 non-null    int64  
 6   CoapplicantIncome  614 non-null    float64
 7   LoanAmount         614 non-null    float64
 8   Loan_Amount_Term   614 non-null    float64
 9   Credit_History     564 non-null    float64
 10  Property_Area      614 non-null    int64  
 11  Loan_Status        614 non-null    int64  
dtypes: float64(5), int64(7)
memory usage: 57.7 KB


In [53]:
for col in df_clean.columns:
  print(col, df_clean[col].unique())

Gender [1 0]
Married [1 0]
Dependents [ 0.  1.  2. nan]
Education [1 0]
Self_Employed [0 1]
ApplicantIncome [ 5849  4583  3000  2583  6000  5417  2333  3036  4006 12841  3200  2500
  3073  1853  1299  4950  3596  3510  4887  2600  7660  5955  3365  3717
  9560  2799  4226  1442  3750  4166  3167  4692  3500 12500  2275  1828
  3667  3748  3600  1800  2400  3941  4695  3410  5649  5821  2645  4000
  1928  3086  4230  4616 11500  2708  2132  3366  8080  3357  3029  2609
  4945  5726 10750  7100  4300  3208  1875  4755  5266  1000  3333  3846
  2395  1378  3988  2366  8566  5695  2958  6250  3273  4133  3620  6782
  2484  1977  4188  1759  4288  4843 13650  4652  3816  3052 11417  7333
  3800  2071  5316  2929  3572  7451  5050 14583  2214  5568 10408  5667
  2137  2957  3692 23803  3865 10513  6080 20166  2014  2718  3459  4895
  3316 14999  4200  5042  6950  2698 11757  2330 14866  1538 10000  4860
  6277  2577  9166  2281  3254 39999  9538  2980  1863  7933  3089  4167
  9323  3707  24

In [64]:
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

print(X.shape)
print(y.shape)

validation_size = 0.20
seed = 123

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=validation_size, random_state=seed)
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

(614, 12)
(614,)


((491, 12), (123, 12), (491,), (123,))

In [66]:
sm = SMOTENC(categorical_features=[0,1,2,3,4,5,11],random_state=seed)
X_res, y_res = sm.fit_resample(X_train, Y_train)

print(X_res.shape, y_res.shape)
print(y_res.value_counts())

model = LogisticRegression(random_state=seed,max_iter=1000)
model.fit(X_res, y_res)

predictions_Smote = model.predict(X_test)

ValueError: Input contains NaN.